# Custom LLM Fine-Tuning on Kaggle GPU

This notebook fine-tunes Phi-3 Mini with LoRA for free on Kaggle GPU.

## Install Dependencies

In [ ]:
!pip install -q torch transformers peft accelerate datasets trl aiohttp beautifulsoup4

## Import Libraries

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
from datasets import Dataset
import json
import aiohttp
import asyncio
from bs4 import BeautifulSoup
import logging
from datetime import datetime
import os
from sklearn.model_selection import train_test_split

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Check GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## Step 1: Collect Training Data

In [ ]:
# Collect data
print("Collecting training data...")
wiki_data = await collect_wikipedia_data(50)
hn_data = await collect_hacker_news_data(50)
all_data = wiki_data + hn_data
print(f"Collected {len(all_data)} samples")

# Transform data to instruction format for training
train_data = []
for item in all_data:
    train_data.append({
        "instruction": f"Summarize or explain: {item['title']}",
        "input": "",
        "output": item['content']
    })

# Split into train and validation sets
train_data, val_data = train_test_split(train_data, test_size=0.2, random_state=42)
print(f"Train samples: {len(train_data)}, Val samples: {len(val_data)}")

## Step 2: Preprocess Data

In [ ]:
# Note: Mistral 7B loading removed - using Phi-3 Mini instead (see next cell)

## Step 3: Load Model and Tokenizer

In [ ]:
# Load Phi-3 Mini model
model_name = "microsoft/Phi-3-mini-4k-instruct"

print(f"Loading model: {model_name}")

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

print("Model loaded successfully")

## Step 4: Apply LoRA

In [ ]:
# LoRA configuration
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# Apply LoRA
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## Step 5: Prepare Training Dataset

In [ ]:
def format_for_training(data):
    """Format data for training."""
    formatted = []
    for item in data:
        if item['input']:
            text = f"### Instruction:\n{item['instruction']}\n\n### Input:\n{item['input']}\n\n### Response:\n{item['output']}"
        else:
            text = f"### Instruction:\n{item['instruction']}\n\n### Response:\n{item['output']}"
        formatted.append({"text": text})
    return formatted

train_formatted = format_for_training(train_data)
val_formatted = format_for_training(val_data)

train_dataset = Dataset.from_list(train_formatted)
val_dataset = Dataset.from_list(val_formatted)

print(f"Train dataset size: {len(train_dataset)}")
print(f"Val dataset size: {len(val_dataset)}")

## Step 6: Train with LoRA

In [ ]:
# Training arguments
training_args = TrainingArguments(
    output_dir="./model_checkpoints",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_steps=100,
    logging_steps=10,
    save_steps=100,
    eval_steps=100,
    evaluation_strategy="steps",
    save_strategy="steps",
    load_best_model_at_end=True,
    fp16=True,
    report_to=["tensorboard"],
    save_total_limit=2
)

# Create trainer
trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    dataset_text_field="text",
    max_seq_length=512,
    tokenizer=tokenizer,
    args=training_args
)

print("Starting training...")
trainer.train()
print("Training complete!")

## Step 7: Save Model

In [ ]:
# Save fine-tuned model
output_dir = "./fine_tuned_model"
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)

print(f"Model saved to {output_dir}")

# Create zip file for download
import shutil
shutil.make_archive("fine_tuned_model", "zip", output_dir)

print("Model zipped and ready for download!")

## Instructions for Downloading Model

1. Look for `fine_tuned_model.zip` in the output folder
2. Download the zip file
3. Extract it to your local project at `./model_checkpoints/`
4. Update `src/web_server.py` to set `use_local_llm = True`
5. Restart your server to use the custom model